In [1]:
import torch
import torch.nn as nn
import numpy as np
from time import time
from models import get_model
import os
import random
from dataset import get_data
from dataset.data_deal import Simu_Dataset
from environment import Env
from time import time
from torch.utils.data import WeightedRandomSampler
import matplotlib.pyplot as plt
from agent import Agent
from inference import Inference
from environment import Env
import seaborn as sns
from math import ceil
from tqdm import tqdm
from scipy.stats import pearsonr
os.environ["CUDA_VISIBLE_DEVICES"] = "6"

In [2]:
class arg:
    def __init__(self) -> None:
        pass

args = arg()
args.random_seed = int(17)
args.model = 'simple'
args.dataset = 'simu_data'
args.task_id = int(2)
args.data_type = str('dpeak_dependent_complex2')

In [3]:
args.train_lr = 0.003
args.val_test_split = [0.25,0.25]
args.n_feature = int(58) if args.dataset == 'ACIC2016' else int(25)
args.missing_ratio = float(1)
args.disable_cuda = False
args.complete = False
args.pretrain = int(1000)
args.pretrain_sample = str('both')
args.mode = str('double')
args.decay = float(0.999)
args.gamma = float(1)
args.dropout = False
args.batchnorm = False
args.done_action_train = False
args.p = float(0)
args.group_norm = float(0)
args.save_dir = str('result')
args.inf_hidden_sizes = [512,512]
args.policy_hidden_sizes = [32]
args.shared_dim = int(16)
args.target_update_freq = int(100)
args.eps_start = float(1.)
args.eps_end = float(0.1)
args.decay_rate = float(2)
args.n_env = int(32)
args.nsteps = int(4)
args.normalize = True
args.embedded_dim = int(16)
args.lstm_size = int(16)
args.n_shuffle = int(5)
args.r_cost = float(1.0)
args.cost_from_file = False
args.batch_size = int(128)
args.message = str('')
args.buffer_size = int(10000)
random.seed(args.random_seed)
np.random.seed(args.random_seed)
torch.manual_seed(args.random_seed)
if not args.disable_cuda and torch.cuda.is_available():
    args.device = torch.device('cuda')
    torch.cuda.manual_seed(args.random_seed)
else:
    args.device = torch.device('cpu')
args.save_dir = os.path.join(os.getcwd(),args.save_dir)
if args.dataset == 'simu_data':
    args.save_path = args.dataset + '_' + args.data_type
elif args.dataset == 'ACIC2016':
    args.save_path = args.dataset + '_' + str(args.task_id)
else:
    args.save_path = args.dataset
args.save_path = args.save_path + '_cost{}'.format(args.r_cost)
args.save_path = os.path.join(args.save_dir, args.save_path)
args.csv_path = args.save_path
args.save_path = args.save_path + '_seed{}'.format(args.random_seed)
args.data_path = os.path.join(os.getcwd(),'dataset')
if not os.path.exists(args.save_path):
    os.makedirs(args.save_path)


In [4]:
args.X_mode,args.T_mode,args.Y_mode = args.data_type.split('_')
traindata,testdata,valdata = get_data(args)

In [5]:
model = get_model(args)
inf = Inference(model,'T_mode',args,5000)
agent = Agent(model,args,5000)
train_env = Env(args.n_env,traindata,model,args.r_cost)
val_env = Env(args.n_env,valdata,model,args.r_cost)
test_env = Env(args.n_env,testdata,model,args.r_cost)
args.missing_ratio = float(0)
inf.pretrain(traindata,valdata,args,10000,128)

start_pretrain
epoch: 10  train_loss: 8.186230659484863  val_loss: 8.926254272460938
epoch: 20  train_loss: 10.00977611541748  val_loss: 8.867650985717773
epoch: 30  train_loss: 8.288176536560059  val_loss: 8.812682151794434
epoch: 40  train_loss: 7.485810279846191  val_loss: 8.765579223632812
epoch: 50  train_loss: 10.430479049682617  val_loss: 8.688057899475098
epoch: 60  train_loss: 11.040488243103027  val_loss: 8.618792533874512
epoch: 70  train_loss: 10.620628356933594  val_loss: 8.546049118041992
epoch: 80  train_loss: 7.560842514038086  val_loss: 8.461454391479492
epoch: 90  train_loss: 9.719171524047852  val_loss: 8.386685371398926
epoch: 100  train_loss: 9.189820289611816  val_loss: 8.302777290344238
epoch: 110  train_loss: 9.83071517944336  val_loss: 8.211779594421387
epoch: 120  train_loss: 5.361469268798828  val_loss: 8.119593620300293
epoch: 130  train_loss: 6.371245861053467  val_loss: 8.0401029586792
epoch: 140  train_loss: 7.695005893707275  val_loss: 7.931771755218506


In [6]:
traindata.pred_ycf(model)
valdata.pred_ycf(model)
testdata.pred_ycf(model)

In [7]:
features = traindata.features
tau = (traindata.y_fact-traindata.y_cf)*(2*traindata.treatments-1)

In [8]:
cor = np.zeros(features.shape[-1])
for i in range(features.shape[-1]):
    cor[i] = pearsonr(features[:,i].numpy(),tau.numpy())[0]
cor = np.abs(cor)
max_cor = cor.max()
min_cor = cor.min()

In [9]:
for threshold in np.arange(0.1,0.91,0.1):
    inf.replay_buffer.reset()
    inf.val_buffer.reset()
    model.load(os.path.join(args.save_path, "pretrained_best.model"))
    mask = torch.Tensor(cor >= threshold*max_cor+(1-threshold)*min_cor)
    print('threshold:',threshold)
    print('n_feature:',mask.int().sum())
    print('selected_feature:',torch.where(mask))
    inf.replay_buffer.push(list(zip(features*mask,mask.expand_as(features),traindata.treatments,traindata.y_fact,traindata.y_cf)))
    inf.val_buffer.push(list(zip(valdata.features*mask,mask.expand_as(valdata.features),valdata.treatments,valdata.y_fact,valdata.y_cf)))
    inf.train(args,1000)
    incomplete_testdata = Simu_Dataset(testdata.features.clone(),testdata.treatments.clone(),testdata.y_fact.clone(),testdata.y_cf.clone(),testdata.mu.clone())
    incomplete_testdata.features[:,~mask.bool()]=0
    inf.test(incomplete_testdata,args,mask.expand_as(testdata.features))

threshold: 0.1
n_feature: tensor(4)
selected_feature: (tensor([3, 4, 5, 6]),)


epoch:0  val_loss: 16.397172927856445  SAVE
epoch: 10  train_loss: 15.59914493560791  val_loss: 15.950831413269043  SAVE
epoch: 30  train_loss: 13.477649688720703  val_loss: 15.90207290649414  SAVE
epoch: 70  train_loss: 14.872693061828613  val_loss: 15.886800765991211  SAVE
start_inference_test
finish_inference_test
time_use: 0.12197661399841309
mse of tau: tensor(15.0294, device='cuda:0', grad_fn=<DivBackward0>)
loss: tensor(15.1245, device='cuda:0', grad_fn=<DivBackward0>)
mse of y_fact: tensor(7.1918, device='cuda:0', grad_fn=<DivBackward0>)
threshold: 0.2
n_feature: tensor(4)
selected_feature: (tensor([3, 4, 5, 6]),)
epoch:0  val_loss: 15.635167121887207  SAVE
epoch: 10  train_loss: 14.977668762207031  val_loss: 14.983354568481445  SAVE
epoch: 20  train_loss: 14.74538516998291  val_loss: 14.961092948913574  SAVE
epoch: 30  train_loss: 14.391803741455078  val_loss: 14.926830291748047  SAVE
epoch: 60  train_loss: 15.233563423156738  val_loss: 14.892152786254883  SAVE
epoch: 70  trai

In [10]:
inf.replay_buffer.reset()
inf.val_buffer.reset()
model.load(os.path.join(args.save_path, "pretrained_best.model"))
mask = torch.Tensor([1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0])
print('threshold:',threshold)
print('n_feature:',mask.int().sum())
print('selected_feature:',torch.where(mask))
inf.replay_buffer.push(list(zip(features*mask,mask.expand_as(features),traindata.treatments,traindata.y_fact,traindata.y_cf)))
inf.val_buffer.push(list(zip(valdata.features*mask,mask.expand_as(valdata.features),valdata.treatments,valdata.y_fact,valdata.y_cf)))
inf.train(args,1000)
incomplete_testdata = Simu_Dataset(testdata.features.clone(),testdata.treatments.clone(),testdata.y_fact.clone(),testdata.y_cf.clone(),testdata.mu.clone())
incomplete_testdata.features[:,~mask.bool()]=0
inf.test(incomplete_testdata,args,mask.expand_as(testdata.features))

threshold: 0.9
n_feature: tensor(7)
selected_feature: (tensor([0, 1, 2, 3, 4, 5, 6]),)
epoch:0  val_loss: 3.840153217315674  SAVE
epoch: 10  train_loss: 1.2247984409332275  val_loss: 1.1221559047698975  SAVE
epoch: 20  train_loss: 0.7973454594612122  val_loss: 0.7529621720314026  SAVE
epoch: 30  train_loss: 0.7303307056427002  val_loss: 0.6310313940048218  SAVE
epoch: 40  train_loss: 0.5519970059394836  val_loss: 0.5680850744247437  SAVE
epoch: 50  train_loss: 0.5896909236907959  val_loss: 0.5269761681556702  SAVE
epoch: 60  train_loss: 0.5746840238571167  val_loss: 0.5042385458946228  SAVE
epoch: 70  train_loss: 0.4708055257797241  val_loss: 0.48230046033859253  SAVE
epoch: 80  train_loss: 0.49577200412750244  val_loss: 0.4678541421890259  SAVE
epoch: 90  train_loss: 0.5807662010192871  val_loss: 0.45882922410964966  SAVE
epoch: 100  train_loss: 0.45662498474121094  val_loss: 0.4454892873764038  SAVE
epoch: 110  train_loss: 0.41353169083595276  val_loss: 0.43870288133621216  SAVE
epoc

(tensor(0.2408, device='cuda:0', grad_fn=<DivBackward0>),
 tensor(0.0969, device='cuda:0', grad_fn=<DivBackward0>))